# ARS Session 3 — Adaptive Steering Router

This experiment compares three isolated arms on the same deterministic GSM8K subset: untouched Phi-2, the fixed V2 checkpoint, and a new question-conditioned router. It does not modify the Session 2 notebook or checkpoint. Phi-2 remains frozen.

Attach the verified V2 checkpoint and its p99 calibration when available. Existing rollout shards are reused. If they are absent, the notebook runs Gate 0 and collects fresh shards. If V2 artifacts are absent, it can calibrate and train a replacement V2 arm from those rollouts. Phase failures are recorded so the notebook reaches its final status cell.


## 1. Execution configuration

Edit the artifact paths to match your Kaggle input dataset.


In [ ]:
# ==========================================
# EXECUTION & SCALE CONFIGURATION
# ==========================================
import os
from pathlib import Path
from datetime import datetime

SMOKE_TEST = False
AUTO_COLLECT_IF_MISSING = True
AUTO_BOOTSTRAP_V2_IF_MISSING = True
ATTENTION_IMPLEMENTATION = "sdpa"
ARTIFACT_INPUT_DIR = Path("/kaggle/input/ars-session2-artifacts")
V2_CHECKPOINT = ARTIFACT_INPUT_DIR / "steernet_v2.pt"
V2_NORM_CALIBRATION = ARTIFACT_INPUT_DIR / "norms_v2.json"
ROLLOUT_DIR = Path("/kaggle/input/ars-session1-rollouts")
V2_ROUTE = [12, 15, 18, 21]
CANDIDATE_LAYERS = [8, 11, 12, 13, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("/kaggle/working/ars_session3") / RUN_ID
BOOTSTRAP_V2_CHECKPOINT = OUTPUT_DIR / "session2_steernet_v2.pt"
BOOTSTRAP_V2_NORM_CALIBRATION = OUTPUT_DIR / "session2_norms_v2.json"
BANK_CHECKPOINT = OUTPUT_DIR / "session3_steering_bank.pt"
ROUTER_CHECKPOINT = OUTPUT_DIR / "session3_router.pt"
EXPANDED_NORM_CALIBRATION = OUTPUT_DIR / "candidate_norms_p99.json"

if SMOKE_TEST:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 24, 10, 40, 2
    V2_BOOTSTRAP_EPOCHS = 1
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 1, 40
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 12, 1
    BENCH_N_TEST, BENCH_SHOTS = 24, [0]
else:
    CALIB_N, GATE0_QUESTIONS, COLLECTION_QUESTIONS, COLLECTION_K = 128, 60, 2000, 8
    V2_BOOTSTRAP_EPOCHS = 2
    BANK_EPOCHS, MAX_RFT_QUESTIONS = 2, 2000
    MAX_ROUTER_ROWS, ROUTER_EPOCHS = 512, 2
    BENCH_N_TEST, BENCH_SHOTS = 800, [0, 2]

print({"smoke_test": SMOKE_TEST, "fixed_route": V2_ROUTE,
       "candidate_layers": CANDIDATE_LAYERS, "calibration_n": CALIB_N,
       "gate0_n": GATE0_QUESTIONS, "auto_bootstrap_v2": AUTO_BOOTSTRAP_V2_IF_MISSING,
       "bank_epochs": BANK_EPOCHS, "router_epochs": ROUTER_EPOCHS,
       "benchmark_n": BENCH_N_TEST, "shots": BENCH_SHOTS,
       "output_dir": str(OUTPUT_DIR)}, flush=True)


## 2. Kaggle checkout and live progress

Training uses one visible T4 at a time for predictable memory use. If collection is needed, its two workers each get one GPU. The stream helper forwards tqdm refreshes to the cell immediately.


In [ ]:
import codecs
import subprocess
import sys
import threading

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

def _stream_child(prefix, command, env=None):
    """Stream progress refreshes and return the child status."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, bufsize=0)
    pending = bytearray()
    while True:
        value = process.stdout.read(1)
        if not value: break
        if value in (b"\r", b"\n"):
            if pending:
                print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
                pending.clear()
        else:
            pending.extend(value)
    if pending: print(prefix + codecs.decode(pending, "utf-8", errors="replace"), flush=True)
    return process.wait()

def run_live(command, *, env=None, prefix=""):
    code = _stream_child(prefix, command, env)
    if code: raise RuntimeError(f"{prefix}child exited with code {code}: {command}")

def run_parallel_live(jobs):
    results = {}
    def worker(job_id, prefix, command, env): results[job_id] = _stream_child(prefix, command, env)
    threads = [threading.Thread(target=worker, args=(i, *job), daemon=True) for i, job in enumerate(jobs)]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    failures = [f"job {i} exited with code {value}" for i, value in results.items() if value]
    if failures: raise RuntimeError("; ".join(failures))

print(f"Repository ready: {REPO_DIR} ({BRANCH})", flush=True)


## 3. Runtime, inputs, and phase tracking

Failures are recorded in the phase status dictionary. Missing checkpoints or data skip dependent phases while allowing the final summary to run.


In [ ]:
import json
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PHASE_STATUS = {name: "pending" for name in (
    "runtime", "tests", "inputs", "rollouts", "v2_bootstrap", "norm_calibration",
    "steering_bank", "router", "benchmark")}
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True, check=True)
    visible_gpus = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    PHASE_STATUS["runtime"] = "passed" if visible_gpus else "no_gpu_detected"
    print("Visible GPUs:", visible_gpus, flush=True)
except Exception as exc:
    visible_gpus = []
    PHASE_STATUS["runtime"] = f"failed: {type(exc).__name__}: {exc}"
    print(PHASE_STATUS["runtime"], flush=True)
try:
    result = subprocess.run([sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"], check=False)
    PHASE_STATUS["tests"] = "passed" if result.returncode == 0 else f"failed_exit_{result.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Tests:", PHASE_STATUS["tests"], flush=True)

if not V2_CHECKPOINT.is_file():
    found = list(Path("/kaggle/input").glob("**/steernet_v2.pt"))
    if found: V2_CHECKPOINT = found[0]
def has_v2_p99_norms(path):
    try:
        calibration = json.loads(path.read_text(encoding="utf-8"))
        layers = calibration.get("layers", {})
        return all(float(layers.get(str(layer), {}).get("p99_norm", 0.0)) > 0 for layer in V2_ROUTE)
    except (OSError, ValueError, TypeError, AttributeError):
        return False

if not has_v2_p99_norms(V2_NORM_CALIBRATION):
    norm_candidates = []
    for filename in ("norms_v2.json", "norms_calibration.json"):
        norm_candidates.extend(Path("/kaggle/input").glob(f"**/{filename}"))
    V2_NORM_CALIBRATION = next((path for path in norm_candidates if has_v2_p99_norms(path)), V2_NORM_CALIBRATION)
PHASE_STATUS["inputs"] = "ready" if V2_CHECKPOINT.is_file() and has_v2_p99_norms(V2_NORM_CALIBRATION) else ("bootstrap_required" if AUTO_BOOTSTRAP_V2_IF_MISSING else "missing_v2_checkpoint_or_norms")
print({"checkpoint": str(V2_CHECKPOINT), "checkpoint_exists": V2_CHECKPOINT.is_file(),
       "v2_norms": str(V2_NORM_CALIBRATION), "v2_norms_exist": V2_NORM_CALIBRATION.is_file()}, flush=True)
def find_complete_rollout_pair(search_roots):
    candidates = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.glob("**/shard[01].jsonl"):
            candidates.setdefault(path.parent.resolve(), {})[path.name] = path
    complete = [pair for pair in candidates.values() if {"shard0.jsonl", "shard1.jsonl"}.issubset(pair)]
    if not complete:
        return []
    complete.sort(key=lambda pair: max(pair[name].stat().st_mtime for name in ("shard0.jsonl", "shard1.jsonl")), reverse=True)
    pair = complete[0]
    return [pair["shard0.jsonl"], pair["shard1.jsonl"]]
ROLLOUT_FILES = find_complete_rollout_pair([ROLLOUT_DIR, Path("/kaggle/working")])
print("Rollout files:", [str(path) for path in ROLLOUT_FILES], flush=True)

PHASE_STATUS["v2_bootstrap"] = "not_needed" if PHASE_STATUS["inputs"] == "ready" else "pending"


## 4. Reuse rollouts or collect if none are attached

Existing verified shards are reused. Only when none are found does the optional two-GPU collection run create new shards in this run directory.


In [ ]:
if len(ROLLOUT_FILES) >= 2:
    PHASE_STATUS["rollouts"] = "reused_existing_shards"
elif AUTO_COLLECT_IF_MISSING:
    try:
        rollout_output = OUTPUT_DIR / "rollouts"
        rollout_output.mkdir(parents=True, exist_ok=True)
        gate0_command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "gate0",
                         "--n_questions", str(GATE0_QUESTIONS), "--k", str(COLLECTION_K),
                         "--temperature", "0.7", "--gate0_nonblocking",
                         "--output_dir", str(rollout_output)]
        gate0_env = os.environ.copy()
        gate0_env["CUDA_VISIBLE_DEVICES"] = "0"
        print(f"Running diagnostic Gate 0 on {GATE0_QUESTIONS} questions before fresh collection", flush=True)
        run_live(gate0_command, env=gate0_env, prefix="[Gate 0] ")
        gate0_stats = json.loads((rollout_output / "gate0_stats.json").read_text(encoding="utf-8"))
        print({key: gate0_stats.get(key) for key in ("n", "pass_at_k_lenient", "greedy_lenient_accuracy", "quality_gate_passed")}, flush=True)
        jobs = []
        for shard, gpu in ((0, "0"), (1, "1")):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = gpu
            command = [sys.executable, "-u", "scripts/run_rft_pipeline.py", "--stage", "collect",
                       "--n_questions", str(COLLECTION_QUESTIONS), "--k", str(COLLECTION_K),
                       "--temperature", "0.7", "--n_shards", "2", "--shard", str(shard),
                       "--allow_low_yield", "--output_dir", str(rollout_output)]
            jobs.append((f"[GPU{gpu}/shard{shard}] ", command, env))
        run_parallel_live(jobs)
        ROLLOUT_FILES = find_complete_rollout_pair([rollout_output])
        PHASE_STATUS["rollouts"] = "collected" if len(ROLLOUT_FILES) == 2 else "collection_outputs_missing"
    except Exception as exc:
        PHASE_STATUS["rollouts"] = f"failed: {type(exc).__name__}: {exc}"
        print("Rollout phase failed; dependent phases will be skipped:", exc, flush=True)
else:
    PHASE_STATUS["rollouts"] = "missing_and_auto_collection_disabled"
print("Rollout status:", PHASE_STATUS["rollouts"], flush=True)


## 5. Rebuild missing V2 artifacts
When no fixed V2 checkpoint and p99 norms are attached, calibrate and train this comparison arm from the same positive rollout shards. This replacement is recorded separately and does not overwrite the original artifact.

In [ ]:
if PHASE_STATUS["inputs"] == "bootstrap_required":
    if PHASE_STATUS["rollouts"] in {"reused_existing_shards", "collected"} and len(ROLLOUT_FILES) == 2:
        try:
            v2_calibration_command = [sys.executable, "-u", "scripts/calibrate_norms.py",
                                      "--n_samples", str(CALIB_N), "--seed", "42",
                                      "--rollout_files", *map(str, ROLLOUT_FILES),
                                      "--candidate_layers", *map(str, V2_ROUTE),
                                      "--max_length", "768", "--output", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                      "--device", "cuda"]
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = "0"
            run_live(v2_calibration_command, env=env, prefix="[V2 norms] ")
            if not BOOTSTRAP_V2_NORM_CALIBRATION.is_file():
                raise FileNotFoundError(f"V2 norm calibration was not produced: {BOOTSTRAP_V2_NORM_CALIBRATION}")
            V2_NORM_CALIBRATION = BOOTSTRAP_V2_NORM_CALIBRATION
            if V2_CHECKPOINT.is_file():
                PHASE_STATUS["v2_bootstrap"] = "checkpoint_reused_norms_recalibrated"
            else:
                v2_train_output = OUTPUT_DIR / "v2_bootstrap"
                v2_train_output.mkdir(parents=True, exist_ok=True)
                v2_train_command = [sys.executable, "-u", "scripts/run_rft_pipeline.py",
                                    "--stage", "train", "--skip_collection", "--steernet_arch", "v2",
                                    "--norm_calibration", str(BOOTSTRAP_V2_NORM_CALIBRATION),
                                    "--rollout_dir", str(ROLLOUT_FILES[0].parent),
                                    "--epochs", str(V2_BOOTSTRAP_EPOCHS),
                                    "--checkpoint", str(BOOTSTRAP_V2_CHECKPOINT),
                                    "--output_dir", str(v2_train_output)]
                run_live(v2_train_command, env=env, prefix="[V2 bootstrap] ")
                if not BOOTSTRAP_V2_CHECKPOINT.is_file():
                    raise FileNotFoundError(f"V2 bootstrap did not produce a checkpoint: {BOOTSTRAP_V2_CHECKPOINT}")
                V2_CHECKPOINT = BOOTSTRAP_V2_CHECKPOINT
                PHASE_STATUS["v2_bootstrap"] = "trained_from_rollouts"
            PHASE_STATUS["inputs"] = "bootstrapped"
        except Exception as exc:
            PHASE_STATUS["v2_bootstrap"] = f"failed: {type(exc).__name__}: {exc}"
            PHASE_STATUS["inputs"] = "bootstrap_failed"
            print("V2 bootstrap failed; dependent training will be skipped, but the notebook will continue:", exc, flush=True)
    else:
        PHASE_STATUS["v2_bootstrap"] = "skipped_no_rollouts"
        PHASE_STATUS["inputs"] = "bootstrap_failed"
else:
    PHASE_STATUS["v2_bootstrap"] = "not_needed" if PHASE_STATUS["inputs"] == "ready" else "disabled"
print("V2 bootstrap status:", PHASE_STATUS["v2_bootstrap"], flush=True)


## 5. Calibrate candidate p99 norms

This uses rollout questions only. The original V2 norm file remains separate so the fixed V2 arm retains its calibrated caps.


In [ ]:
if PHASE_STATUS["rollouts"] in {"reused_existing_shards", "collected"} and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [sys.executable, "-u", "scripts/calibrate_norms.py", "--n_samples", str(CALIB_N),
                   "--seed", "42", "--rollout_files", *[str(path) for path in ROLLOUT_FILES],
                   "--candidate_layers", *[str(layer) for layer in CANDIDATE_LAYERS],
                   "--max_length", "768", "--output", str(EXPANDED_NORM_CALIBRATION), "--device", "cuda"]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[norms] ")
        PHASE_STATUS["norm_calibration"] = "passed" if EXPANDED_NORM_CALIBRATION.is_file() else "output_missing"
    except Exception as exc:
        PHASE_STATUS["norm_calibration"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["norm_calibration"], flush=True)
else:
    PHASE_STATUS["norm_calibration"] = "skipped_missing_prerequisites"


## 6. Train variable-subset steering adapters

V2 initializes its four matching layers; the remaining layers start as new V2-style adapters. Positive-completion NLL trains sampled subsets with k balanced from 0 to 4 and alpha sampled in [1, 2]. Phi-2 remains frozen and the learning rate stays at 1.5e-4.


In [ ]:
if PHASE_STATUS["norm_calibration"] == "passed" and PHASE_STATUS["inputs"] in {"ready", "bootstrapped"} and ROLLOUT_FILES:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_bank",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--rollout_files", *map(str, ROLLOUT_FILES),
                   "--epochs", str(BANK_EPOCHS), "--max_questions", str(MAX_RFT_QUESTIONS)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[bank] ")
        PHASE_STATUS["steering_bank"] = "passed" if BANK_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["steering_bank"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["steering_bank"], flush=True)
else:
    PHASE_STATUS["steering_bank"] = "skipped_prerequisite_failure"


## 7. Train the prompt-conditioned router

The router sees prompt positions only. It learns from paired outcomes of sampled route actions. Its action set includes no steering, one to four layers, and bounded alpha choices from 1.0 to 2.0.


In [ ]:
if PHASE_STATUS["steering_bank"] == "passed" and ROLLOUT_FILES:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "train_router",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
                   "--rollout_files", *map(str, ROLLOUT_FILES), "--epochs", str(ROUTER_EPOCHS),
                   "--max_questions", str(MAX_RFT_QUESTIONS), "--max_router_rows", str(MAX_ROUTER_ROWS)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[router] ")
        PHASE_STATUS["router"] = "passed" if ROUTER_CHECKPOINT.is_file() else "checkpoint_missing"
    except Exception as exc:
        PHASE_STATUS["router"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["router"], flush=True)
else:
    PHASE_STATUS["router"] = "skipped_prerequisite_failure"


## 8. Compare the three arms on one paired subset

All arms share the same seeded questions, shot conditions, greedy decoding, and repetition penalty 1.0. The output directory contains arm-level JSONL predictions and paired summaries.


In [ ]:
if PHASE_STATUS["inputs"] in {"ready", "bootstrapped"}:
    try:
        command = [sys.executable, "-u", "scripts/run_session3_router.py", "--stage", "benchmark",
                   "--output_dir", str(OUTPUT_DIR), "--candidate_layers", *map(str, CANDIDATE_LAYERS),
                   "--norm_calibration", str(EXPANDED_NORM_CALIBRATION),
                   "--v2_norm_calibration", str(V2_NORM_CALIBRATION), "--v2_checkpoint", str(V2_CHECKPOINT),
                   "--bank_checkpoint", str(BANK_CHECKPOINT), "--router_checkpoint", str(ROUTER_CHECKPOINT),
                   "--bench_n", str(BENCH_N_TEST), "--bench_seed", "42", "--shots", *map(str, BENCH_SHOTS)]
        env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
        run_live(command, env=env, prefix="[benchmark] ")
        reports = list(OUTPUT_DIR.glob("session3_gsm8k_*.json"))
        if reports:
            report = json.loads(reports[0].read_text(encoding="utf-8"))
            PHASE_STATUS["benchmark"] = "passed_three_arms" if report.get("router_status") == "included" else "passed_baseline_and_v2_router_unavailable"
        else:
            PHASE_STATUS["benchmark"] = "report_missing"
    except Exception as exc:
        PHASE_STATUS["benchmark"] = f"failed: {type(exc).__name__}: {exc}"
        print(PHASE_STATUS["benchmark"], flush=True)
else:
    PHASE_STATUS["benchmark"] = "skipped_missing_v2_artifacts"
print("Benchmark status:", PHASE_STATUS["benchmark"], flush=True)


## 9. Final status and artifacts

A completed notebook does not imply the router improved accuracy. Keep it only if the paired results show useful improvement over both baseline and fixed V2.


In [ ]:
summary = {
    "phase_status": PHASE_STATUS,
    "run_dir": str(OUTPUT_DIR),
    "v2_checkpoint": str(V2_CHECKPOINT),
    "v2_norm_calibration": str(V2_NORM_CALIBRATION),
    "v2_artifact_provenance": PHASE_STATUS.get("v2_bootstrap"),
    "fixed_route": V2_ROUTE,
    "router_candidates": CANDIDATE_LAYERS,
    "bank_checkpoint": str(BANK_CHECKPOINT) if BANK_CHECKPOINT.is_file() else None,
    "router_checkpoint": str(ROUTER_CHECKPOINT) if ROUTER_CHECKPOINT.is_file() else None,
    "benchmark_reports": [str(path) for path in OUTPUT_DIR.glob("session3_gsm8k_*.json")],
    "per_question_files": [str(path) for path in (OUTPUT_DIR / "generations").glob("*.jsonl")] if (OUTPUT_DIR / "generations").exists() else [],
}
(OUTPUT_DIR / "session3_status.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2), flush=True)
